# Milk evaluation lab: placement, motion and Jev

Select a GPU runtime and run setup cells 1–5 once. In cell 6, tick any algorithms
and run the cell. Cell 7 displays results and experiment history.

**Normal update loop:** publish code to GitHub → run source cell 3 → run preflight
cell 5 → select methods/run cell 6 → review cell 7. Source refresh does not recopy
videos or delete existing checkouts. Run data-sync cell 4 again when dataset or
weights change. You control when evaluations run.

The selected Git revision must contain the shared-observation/three-method evaluator.
The current local implementation must be published before the older GitHub main
can run this workflow. Every run records the resolved commit, settings and GPU.

Keep the complete annotated Dataset/videos and Models/milk-vision-v4s.pt under
MyDrive/Yolo_Cloud. GPU accelerates YOLO; Jev uses TypeSafe's API. Configure the
Colab secret TYPESAFE_API_KEY only when using Jev. Never paste it into notebook code.

Fresh predictions are the default; compatible detector observations are reused.
Choose Resume to continue interrupted work. Reports, logs and caches persist in
Drive. Inventory-event agreement is not independently verified visible-motion accuracy.


In [ ]:
# 1. Mount your Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# @title 2. Paths and published code revision
from pathlib import Path

GIT_REF = "main" # @param {type:"string"}
MODEL_NAME = "milk-vision-v4s.pt" # @param {type:"string"}
REPO_URL = "https://github.com/Jae-Zen/milk-vision-scripts.git"
DRIVE_ROOT = Path("/content/drive/MyDrive/Yolo_Cloud")
REPO_DIR = Path("/content/milk_vision_git")
CODE_ROOT = Path("/content/milk_vision_code")
DATA_ROOT = Path("/content/milk_vision_data")
VIDEOS = DATA_ROOT / "Dataset" / "videos"
MODEL = DATA_ROOT / "Models" / MODEL_NAME
RESULTS = DRIVE_ROOT / "results" / "evaluation"


In [ ]:
# 3. Refresh a published revision without deleting source or data
import subprocess
import sys


def git_output(repo, *arguments):
    return subprocess.check_output(["git", "-C", str(repo), *arguments], text=True).strip()


def fetch_revision(repo_url, requested_ref, repo_dir, code_root):
    requested_ref = requested_ref.strip()
    if not requested_ref or requested_ref.startswith("-"):
        raise ValueError("Enter a branch, tag or published commit SHA.")
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--no-checkout", repo_url, str(repo_dir)], check=True)
    if git_output(repo_dir, "remote", "get-url", "origin") != repo_url:
        raise RuntimeError("Existing source repository has another origin; it was not changed.")
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", requested_ref], check=True)
    sha = git_output(repo_dir, "rev-parse", "--verify", "FETCH_HEAD^{commit}")
    checkout = code_root / sha / "Scripts"
    if checkout.exists():
        if git_output(checkout, "rev-parse", "HEAD") != sha or git_output(checkout, "status", "--porcelain"):
            raise RuntimeError("Runtime checkout has edits or another revision; preserve/publish them first. No files were deleted.")
    else:
        checkout.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "-C", str(repo_dir), "worktree", "add", "--detach", str(checkout), sha], check=True)
    return sha, checkout


def require_shared_evaluator(checkout, check_cli=True):
    required = ["milk_vision/algorithms/jev.py", "milk_vision/observations/storage.py", "tools/evaluate.py"]
    if any(not (checkout / name).is_file() for name in required):
        raise RuntimeError("This published revision lacks the shared three-method evaluator. Publish the current local implementation or choose a newer Git ref.")
    if not check_cli:
        return
    help_text = subprocess.check_output([sys.executable, "-m", "tools.evaluate", "--help"], cwd=checkout, text=True)
    for flag in ("--methods", "--cache-dir", "--rerun", "--jev-model"):
        if flag not in help_text:
            raise RuntimeError(f"This revision does not support {flag}; choose a compatible published revision.")


READY_SOURCE_SHA = None
if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or newer is required.")
CODE_SHA, SCRIPTS = fetch_revision(REPO_URL, GIT_REF, REPO_DIR, CODE_ROOT)
require_shared_evaluator(SCRIPTS, check_cli=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(SCRIPTS / "requirements.txt")], check=True)
SOURCE_INFO = {"ref": GIT_REF.strip(), "repository": REPO_URL, "commit": CODE_SHA}
require_shared_evaluator(SCRIPTS)
READY_SOURCE_SHA = CODE_SHA
print("Requested ref:", GIT_REF)
print("Resolved commit:", CODE_SHA)
print("Checkout:", SCRIPTS)


In [ ]:
# 4. Sync inputs once per runtime, or rerun when dataset/weights change
import shutil

source_videos = DRIVE_ROOT / "Dataset" / "videos"
source_model = DRIVE_ROOT / "Models" / MODEL_NAME
for required in (source_videos / "annotations.json", source_model):
    if not required.is_file():
        raise FileNotFoundError(f"Missing {required}; sync the complete annotated dataset and selected model to Drive.")
if not VIDEOS.resolve().is_relative_to(DATA_ROOT.resolve()) or VIDEOS.name != "videos":
    raise ValueError("Unexpected disposable runtime destination")
VIDEOS.parent.mkdir(parents=True, exist_ok=True)
MODEL.parent.mkdir(parents=True, exist_ok=True)
if VIDEOS.exists():
    shutil.rmtree(VIDEOS)
shutil.copytree(source_videos, VIDEOS)
shutil.copy2(source_model, MODEL)
RESULTS.mkdir(parents=True, exist_ok=True)
print("Input copies ready:", VIDEOS, MODEL)
print("Code refresh and run cells do not repeat this data copy.")


In [ ]:
# 5. Lightweight preflight and reusable experiment helpers; no inference/API calls
import hashlib
import json
import os
import re

import torch


def choose_methods(placement, motion, jev):
    selected = [name for name, enabled in (("placement", placement), ("motion", motion), ("jev", jev)) if enabled]
    if not selected:
        raise ValueError("Tick at least one algorithm before running.")
    return selected


def build_evaluation_command(python, videos, model, cache, report, methods,
                             mode="Fresh predictions", include_negatives=True,
                             scenario="", limit=0, jev_model="jev-1.14.0"):
    if not methods or any(name not in ("placement", "motion", "jev") for name in methods):
        raise ValueError("Select placement, motion and/or Jev.")
    if mode not in ("Fresh predictions", "Resume") or isinstance(limit, bool) or int(limit) != limit or limit < 0:
        raise ValueError("Choose a valid run mode and a nonnegative integer limit.")
    command = [python, "-u", "-m", "tools.evaluate", "--root", str(videos),
               "--methods", *methods, "--model", str(model), "--device", "0",
               "--cache-dir", str(cache), "--jev-model", jev_model, "--output", str(report)]
    if mode == "Fresh predictions": command.append("--rerun")
    if include_negatives: command.append("--include-negatives")
    if scenario.strip(): command.extend(["--scenario", scenario.strip()])
    if limit: command.extend(["--limit", str(int(limit))])
    return command


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def select_scope(command, checkout):
    listing = list(command)
    if "--rerun" in listing: listing.remove("--rerun")
    index = listing.index("--output")
    del listing[index:index + 2]
    listing.append("--list")
    selected = subprocess.check_output(listing, cwd=checkout, text=True).splitlines()
    if not selected: raise RuntimeError("No videos match this selection.")
    return selected


def history_rows(results):
    rows = []
    for path in sorted(results.glob("*/metadata.json"), reverse=True):
        metadata = json.loads(path.read_text(encoding="utf-8"))
        report_path = path.parent / "report.json"
        report = json.loads(report_path.read_text(encoding="utf-8")) if report_path.exists() else {}
        completed = len(report.get("results", []))
        planned = metadata["selected_videos"] * len(metadata["methods"])
        failed = sum(bool(item.get("error")) for item in report.get("results", []))
        status = "partial" if completed != planned or report.get("cancelled") else ("processing failures" if failed else "complete")
        evidence = {item["video"]: item.get("details", {}).get("collection", {}).get("observation_sha256") for item in report.get("results", [])}
        evidence_id = hashlib.sha256(json.dumps(evidence, sort_keys=True).encode()).hexdigest()[:12] if completed == planned and all(evidence.values()) else "partial/unknown"
        for method in metadata["methods"]:
            score = report.get("summary", {}).get(method, {}).get("overall", {})
            rows.append({"run": path.parent.name, "label": metadata["label"],
                         "commit": metadata["commit"], "ref": metadata["requested_ref"],
                         "method": method, "status": status, "progress": f"{completed}/{planned}",
                         "net_accuracy": score.get("net_accuracy"), "event_accuracy": score.get("event_sequence_accuracy"),
                         "scope": metadata["scope_sha256"], "evidence": evidence_id,
                         "settings": metadata["settings_sha256"], "gpu": metadata["gpu"],
                         "report": str(report_path)})
    return rows


if not globals().get("READY_SOURCE_SHA") or READY_SOURCE_SHA != CODE_SHA:
    raise RuntimeError("Run the source setup cell successfully first.")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime and reconnect. CPU fallback is disabled.")
if not (VIDEOS / "annotations.json").is_file() or not MODEL.is_file():
    raise RuntimeError("Run the input-sync cell for this dataset and checkpoint first.")
GPU_NAME = torch.cuda.get_device_name(0)
CACHE = DRIVE_ROOT / "cache" / ("observations_" + re.sub(r"[^A-Za-z0-9_-]", "_", GPU_NAME))
print("GPU:", GPU_NAME)
print("Ready code commit:", CODE_SHA)
print("Persistent observations:", CACHE)
print("Use the checkboxes in cell 6, then run that cell. No evaluation has started.")


In [ ]:
# @title 6. Select algorithms and run an experiment
RUN_PLACEMENT = True # @param {type:"boolean"}
RUN_MOTION = True # @param {type:"boolean"}
RUN_JEV = True # @param {type:"boolean"}
RUN_MODE = "Fresh predictions" # @param ["Fresh predictions", "Resume"]
RUN_LABEL = "" # @param {type:"string"}
INCLUDE_NEGATIVES = True # @param {type:"boolean"}
SCENARIO = "" # @param {type:"string"}
LIMIT = 0 # @param {type:"integer"}
JEV_MODEL = "jev-1.14.0" # @param {type:"string"}

import uuid
from datetime import UTC, datetime

METHODS = choose_methods(RUN_PLACEMENT, RUN_MOTION, RUN_JEV)
if not globals().get("READY_SOURCE_SHA") or READY_SOURCE_SHA != CODE_SHA or not globals().get("GPU_NAME"):
    raise RuntimeError("Run source setup and preflight first.")
if SOURCE_INFO["ref"] != GIT_REF.strip() or SOURCE_INFO["repository"] != REPO_URL:
    raise RuntimeError("Code reference changed; refresh source before running.")
if not (VIDEOS / "annotations.json").is_file() or not MODEL.is_file():
    raise RuntimeError("Sync the selected dataset/checkpoint first.")
if git_output(SCRIPTS, "rev-parse", "HEAD") != CODE_SHA or git_output(SCRIPTS, "status", "--porcelain"):
    raise RuntimeError("Checkout changed since setup; publish edits and refresh the source.")
if not torch.cuda.is_available() or torch.cuda.get_device_name(0) != GPU_NAME:
    raise RuntimeError("GPU changed or disconnected; rerun preflight.")
label_slug = re.sub(r"[^A-Za-z0-9_-]", "_", RUN_LABEL.strip())[:60]
run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + CODE_SHA[:8] + "_" + uuid.uuid4().hex[:8]
if label_slug: run_id += "_" + label_slug
RUN_DIR = RESULTS / run_id
REPORT, LOG = RUN_DIR / "report.json", RUN_DIR / "run.log"
command = build_evaluation_command(sys.executable, VIDEOS, MODEL, CACHE, REPORT,
    METHODS, RUN_MODE, INCLUDE_NEGATIVES, SCENARIO, LIMIT, JEV_MODEL)
selected = select_scope(command, SCRIPTS)
# Only selected Jev runs request access to the secret. The value is never printed.
if "jev" in METHODS and not os.environ.get("TYPESAFE_API_KEY", "").strip():
    from google.colab import userdata
    os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")
if "jev" in METHODS and not os.environ.get("TYPESAFE_API_KEY", "").strip():
    raise RuntimeError("Set TYPESAFE_API_KEY in Colab Secrets and enable notebook access.")
settings = {"methods": METHODS, "mode": RUN_MODE, "include_negatives": INCLUDE_NEGATIVES,
            "scenario": SCENARIO, "limit": LIMIT, "detector": MODEL_NAME,
            "detector_sha256": sha256_file(MODEL), "jev_model": JEV_MODEL,
            "device": "0", "detector_confidence": .10, "crossing_confidence": .40,
            "gate_y": .27, "motion": {"window_seconds": .6, "max_gap_seconds": .5,
                "min_speed": .04, "min_displacement": .08, "inward": "down"}}
metadata = {"label": RUN_LABEL, "requested_ref": SOURCE_INFO["ref"], "commit": CODE_SHA,
            "repository": REPO_URL, "selected_videos": len(selected), "selection": selected,
            "gpu": GPU_NAME, "python": sys.version, "torch": torch.__version__,
            "cuda": torch.version.cuda, "settings": settings,
            "manifest_sha256": sha256_file(VIDEOS / "annotations.json")}
metadata["scope_sha256"] = hashlib.sha256(json.dumps({"videos": selected,
    "manifest": metadata["manifest_sha256"], "model": settings["detector_sha256"]}, sort_keys=True).encode()).hexdigest()[:12]
metadata["settings_sha256"] = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:12]
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print(f"Selected: {len(selected)} videos x {len(METHODS)} methods")
print("Commit:", CODE_SHA, "Methods:", METHODS, "Mode:", RUN_MODE)
print("Report:", REPORT)
with LOG.open("x", encoding="utf-8") as log:
    process = subprocess.Popen(command, cwd=SCRIPTS, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        return_code = process.wait()
    except KeyboardInterrupt:
        import signal
        process.send_signal(signal.SIGINT)
        process.wait()
        raise
if return_code not in (0, 1):
    raise RuntimeError(f"Evaluator exited {return_code}; inspect {LOG}")
if not REPORT.is_file():
    raise RuntimeError(f"No report was produced; inspect {LOG}")
print("Finished. Run cell 7 to separate label mismatches from processing failures.")


In [ ]:
# 7. Results and previous experiments (safe to rerun after interruption)
import pandas as pd
from IPython.display import display

history = pd.DataFrame(history_rows(RESULTS))
if history.empty:
    print("No experiment metadata found yet. Run cell 6 to create an experiment.")
else:
    display(history)
    print("Compare like-for-like scope, evidence and settings. A different commit identifies different code.")
    print("Partial rows are not complete benchmark scores; processing failures remain in the accuracy denominator.")
if globals().get("REPORT") and REPORT.is_file():
    report = json.loads(REPORT.read_text(encoding="utf-8"))
    expected = report["selected_videos"] * len(report["methods"])
    completed = len(report["results"])
    print(f"Current report: {completed}/{expected} predictions; cancelled={report['cancelled']}")
    if not report["results"]:
        print("No completed predictions yet. Choose Resume in cell 6 to continue.")
    else:
        comparison = pd.DataFrame({method: values["overall"] for method, values in report["summary"].items()}).T
        display(comparison)
        comparison.to_csv(REPORT.parent / "summary.csv")
        evidence_summary = pd.DataFrame({method: {
            "empty_inputs": values["empty_detections"]["videos"],
            "nonempty_inputs": values["with_detections"]["videos"],
            "empty_net_accuracy": values["empty_detections"]["net_accuracy"],
            "nonempty_net_accuracy": values["with_detections"]["net_accuracy"]}
            for method, values in report["summary"].items()}).T
        display(evidence_summary)
        details = pd.DataFrame(report["results"])
        failures = details[details["error"].notna()]
        mismatches = details[details["error"].isna() & details.apply(
            lambda row: row["predicted_net"] != row["expected_net"] or
            (row["expected_events"] is not None and row["predicted_events"] != row["expected_events"]), axis=1)]
        if not failures.empty: display(failures[["method", "video", "error"]])
        if not mismatches.empty: display(mismatches[["method", "video", "expected_net", "predicted_net", "expected_events", "predicted_events"]])
        print("Inventory annotations are not independent visible-motion labels; none on empty evidence can be appropriate.")
        print("Report:", REPORT)
        if completed != expected or report["cancelled"]: print("PARTIAL: choose Resume in cell 6 to reuse completed work.")
        if not failures.empty: print("PROCESSING FAILURES: inspect the failure table before interpreting scores.")
